In [117]:
gen_report = False

In [118]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 184


/tmp/ipykernel_1174231/1318901905.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_1174231/1318901905.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"


In [119]:
def filter_df(filters, df=df, sort_by=["seed"]):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=sort_by)
    

In [120]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    if 'mutual_play_accuracy' in df:
        max_col = 'mutual_play_accuracy'
    else:
        max_col = 'test_accuracy'
    max_val = pd.to_numeric(df[max_col]).max()

    def highlight_max_row(row):
        if  pd.to_numeric(row[max_col]) == max_val:
            return ['font-weight: bold; background-color: #ffff99'] * len(row)
        else:
            return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01 else x)
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#4b0082",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [121]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [122]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "agent_a_training_mode"], max_col="test_time_training_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[pd.to_numeric(section[max_col]).idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols)


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [123]:
clear()

---

In [124]:
add_heading(1, "EXP1: ")

write(
"""
"""
)

In [125]:
scaling_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_training_accuracy",
    "test_time_self_play_accuracy",
    "path",
]

scaling_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",


]

adapt_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_training_accuracy",
    "test_time_self_play_accuracy",
    "path",
]

adapt_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

# Shape - Euclidean

## Backbone

In [126]:
add_heading(2, "ShapeWorld")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [127]:
final = extract_maxes(res, cols=["agent_a_training_mode", "message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,shape,euclidean,"[1, 2, 3, 4]",5,one_shape,-,1e-05,1,0.765,None,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...
3,True,shape,euclidean,"[2, 3, 4]",5,one_shape,-,1e-05,1,0.746,None,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,"[3, 4]",5,one_shape,-,1e-05,1,0.82,None,20251221_0128_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,[4],5,one_shape,-,1e-05,1,0.74,None,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [128]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "dataset_tt": "two_shape",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
160,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,-,1e-01,10,0.591,0.489,20251223_0952_bs32_vocab10_repr1024_lr1_0.001_...
168,True,shape,euclidean,"[2, 3, 4]",5,two_shape,-,1e-01,10,0.591,0.53,20251223_0934_bs32_vocab10_repr1024_lr1_0.001_...
161,True,shape,euclidean,[4],5,two_shape,-,1e-01,10,0.612,0.539,20251223_0935_bs32_vocab10_repr1024_lr1_0.001_...


## Scaling

In [129]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [130]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,scaling,1e-02,30,0.592,0.495,20251223_0218_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,"[2, 3, 4]",5,two_shape,scaling,1e-01,20,0.592,0.545,20251223_0329_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,[4],5,two_shape,scaling,1e-02,30,0.613,0.54,20251223_0549_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [131]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "adaptation"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [132]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,shape,euclidean,"[1, 2, 3, 4]",5,two_shape,adaptation,1e-01,30,0.591,0.489,20251223_0309_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,euclidean,"[2, 3, 4]",5,two_shape,adaptation,1e-02,30,0.59,0.537,20251223_0447_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,euclidean,[4],5,two_shape,adaptation,1e-01,30,0.614,0.534,20251223_0641_bs32_vocab10_repr1024_lr1_0.001_...


# Shape - Cosine

## Backbone

In [133]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [134]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,shape,cosine,"[1, 2, 3, 4]",5,one_shape,-,1e-05,1,0.853,None,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...
3,True,shape,cosine,"[2, 3, 4]",5,one_shape,-,1e-05,1,0.891,None,20251220_0820_bs32_vocab10_repr1024_lr1_0.0001...
1,True,shape,cosine,"[3, 4]",5,one_shape,-,1e-05,1,0.891,None,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,cosine,[4],5,one_shape,-,1e-05,1,0.862,None,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [135]:
add_heading(3, "")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "dataset_tt": "two_shape",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
174,True,shape,cosine,"[1, 2, 3, 4]",5,two_shape,-,1e-02,30,0.514,0.511,20251222_2114_bs32_vocab10_repr1024_lr1_0.001_...
9,True,shape,cosine,"[2, 3, 4]",5,two_shape,-,1e-01,10,0.45,0.386,20251222_1850_bs32_vocab10_repr1024_lr1_0.001_...
182,True,shape,cosine,"[3, 4]",5,two_shape,-,1e-02,10,0.473,0.414,20251223_0933_bs32_vocab10_repr1024_lr1_0.001_...
114,True,shape,cosine,[4],5,two_shape,-,1e-02,10,0.595,0.525,20251223_0932_bs32_vocab10_repr1024_lr1_0.001_...


## Scaling

In [136]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [137]:
final = extract_maxes(res, cols=["message_length"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,shape,cosine,"[1, 2, 3, 4]",5,two_shape,scaling,1e-02,10,0.527,0.52,20251222_1916_bs32_vocab10_repr1024_lr1_0.001_...
3,True,shape,cosine,"[2, 3, 4]",5,two_shape,scaling,1e-01,20,0.463,0.421,20251222_1121_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,cosine,"[3, 4]",5,two_shape,scaling,1e-01,10,0.494,0.437,20251223_0017_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,cosine,[4],5,two_shape,scaling,1e-02,30,0.603,0.535,20251222_2218_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [138]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "adaptation"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[adapt_cols]

In [139]:
final = extract_maxes(res, cols=["message_length"])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,shape,cosine,"[1, 2, 3, 4]",5,adaptation,1e-05,5,0.529,0.536,20251222_2010_bs32_vocab10_repr1024_lr1_0.001_...
3,True,shape,cosine,"[2, 3, 4]",5,adaptation,1e-04,10,0.454,0.393,20251222_1315_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape,cosine,"[3, 4]",5,adaptation,1e-04,20,0.481,0.426,20251223_0123_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape,cosine,[4],5,adaptation,1e-04,10,0.602,0.543,20251222_2328_bs32_vocab10_repr1024_lr1_0.001_...


# MNIST

## Backbone

In [140]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
18,True,mnist1,cosine,"[1, 2, 3, 4]",5,mnist1,-,1e-02,100,0.925,0.848,20251225_0503_bs32_vocab10_repr192_lr1_0.0001_...
173,True,mnist1,cosine,"[2, 3, 4]",5,mnist1,-,1e-02,100,0.944,0.845,20251225_0543_bs32_vocab10_repr192_lr1_0.0001_...
36,True,mnist1,cosine,"[3, 4]",2,mnist1,-,1e-01,10,0.443,0.347,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...
47,True,mnist1,cosine,"[3, 4]",4,mnist2,-,1e-02,100,0.439,0.464,20251224_1857_bs32_vocab10_repr192_lr1_0.0001_...
16,True,mnist1,cosine,"[3, 4]",6,mnist2,-,1e-01,10,0.5,0.534,20251223_1959_bs32_vocab10_repr192_lr1_0.0001_...
29,True,mnist1,cosine,"[3, 4]",8,mnist2,-,1e-02,100,0.495,0.518,20251224_0942_bs32_vocab10_repr192_lr1_0.0001_...
50,True,mnist1,cosine,"[3, 4]",10,mnist2,-,1e-02,100,0.436,0.46,20251224_1856_bs32_vocab10_repr192_lr1_0.0001_...
14,True,mnist1,cosine,[4],5,mnist1,-,1e-02,100,0.676,0.647,20251225_0626_bs32_vocab10_repr192_lr1_0.0001_...


## Baseline

In [141]:
add_heading(3, "")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "dataset_tt": "mnist2",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
47,True,mnist1,cosine,"[3, 4]",4,mnist2,-,1e-02,100,0.439,0.464,20251224_1857_bs32_vocab10_repr192_lr1_0.0001_...
16,True,mnist1,cosine,"[3, 4]",6,mnist2,-,1e-01,10,0.5,0.534,20251223_1959_bs32_vocab10_repr192_lr1_0.0001_...
29,True,mnist1,cosine,"[3, 4]",8,mnist2,-,1e-02,100,0.495,0.518,20251224_0942_bs32_vocab10_repr192_lr1_0.0001_...
50,True,mnist1,cosine,"[3, 4]",10,mnist2,-,1e-02,100,0.436,0.46,20251224_1856_bs32_vocab10_repr192_lr1_0.0001_...


## Scaling

In [142]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "message_length_tt", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [143]:
final = extract_maxes(res, cols=["message_length", "message_length_tt"])
final[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,mnist1,cosine,"[3, 4]",4,mnist2,scaling,1e-01,100,1e-02,1e-02,20251224_1329_bs32_vocab1_repr192_lr1_0.0001_l...
1,True,mnist1,cosine,"[3, 4]",5,mnist2,scaling,1e-02,100,0.494,0.529,20251224_2337_bs32_vocab10_repr192_lr1_0.0001_...
2,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-02,60,0.522,0.551,20251223_2042_bs32_vocab10_repr192_lr1_0.0001_...
3,True,mnist1,cosine,"[3, 4]",8,mnist2,scaling,1e-02,100,0.524,0.545,20251223_2157_bs32_vocab10_repr192_lr1_0.0001_...
4,True,mnist1,cosine,"[3, 4]",10,mnist2,scaling,1e-02,100,0.474,0.511,20251224_1034_bs32_vocab10_repr192_lr1_0.0001_...
5,True,mnist1,cosine,"[3, 4]",12,mnist2,scaling,1e-02,100,0.397,0.474,20251225_0102_bs32_vocab10_repr192_lr1_0.0001_...
6,True,mnist1,cosine,"[3, 4]",14,mnist2,scaling,1e-02,100,0.328,0.449,20251225_0226_bs32_vocab10_repr192_lr1_0.0001_...


## Adaptation

In [144]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "adaptation"
}, sort_by=["message_length", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[scaling_cols_report])

# res[adapt_cols]

In [145]:
final = extract_maxes(res, cols=["message_length", "message_length_tt"])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_training_accuracy,test_time_self_play_accuracy,path
0,True,mnist1,cosine,"[3, 4]",4,adaptation,1e-05,10,0.445,0.476,20251224_1542_bs32_vocab10_repr192_lr1_0.0001_...
1,True,mnist1,cosine,"[3, 4]",5,adaptation,1e-04,5,0.491,0.52,20251225_0402_bs32_vocab10_repr192_lr1_0.0001_...
2,True,mnist1,cosine,"[3, 4]",6,adaptation,1e-04,5,0.512,0.562,20251223_1844_bs32_vocab10_repr192_lr1_0.0001_...
3,True,mnist1,cosine,"[3, 4]",8,adaptation,1e-04,5,0.511,0.54,20251224_0612_bs32_vocab10_repr192_lr1_0.0001_...
4,True,mnist1,cosine,"[3, 4]",10,adaptation,1e-04,5,0.461,0.5,20251224_1401_bs32_vocab10_repr192_lr1_0.0001_...
5,True,mnist1,cosine,"[3, 4]",12,adaptation,1e-04,5,0.385,0.478,20251225_0414_bs32_vocab10_repr192_lr1_0.0001_...
6,True,mnist1,cosine,"[3, 4]",14,adaptation,1e-04,5,0.307,0.463,20251225_0437_bs32_vocab10_repr192_lr1_0.0001_...
